# Этап C **вручную**: замер, проба, генерация по ячейкам

> Эта тетрадь — для работы руками, когда вы сидите рядом и смотрите
> на вывод. Если нужно **запустить и уйти**, берите
> [`kaggle_phase_c_commit.ipynb`](kaggle_phase_c_commit.ipynb):
> `Save & Run All` стартует в чистой сессии, где нет ни vLLM,
> ни весов, ни нарезки, и тетрадь обязана поставить всё сама.
> Эта — не обязана и потому для ухода не годится.

Перед запуском:

* **+ Add Input** → датасет `ru-fin-retrieval` (код и корпус) **и** датасет
  с набором запросов (`queries.jsonl`, `split.json`, `pool_candidates.tsv`);
* **Settings** → **Accelerator** → **GPU T4 ×2**, интернет включён;
* **токен HuggingFace подключён к этой тетради.** Секреты Kaggle
  привязываются к конкретной тетради через браузер и новым ядром
  не наследуются. Один раз проект на этом потерял два часа квоты:
  без токена скачивание весов режется по скорости и может встать намертво.

**Запускать отсюда, из браузера, а не `scripts/kaggle_run.py`.**
В клиенте Kaggle нет поля для секретов вообще: ядро, созданное
через API, получить `HF_TOKEN` не может, и проверено это бесплатным
прогоном без ускорителя. Через API считается только то, чему токен
не нужен, — этап B без ускорителя.

Порядок действий целиком — `kaggle/README.md`.

## 1. Карта и токен

Должна быть видна Tesla T4. Токен проверяется до всякого долгого прогона.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

from kaggle_secrets import UserSecretsClient
try:
    token = UserSecretsClient().get_secret("HF_TOKEN")
    import os
    os.environ["HF_TOKEN"] = token
    os.environ["HUGGING_FACE_HUB_TOKEN"] = token
    print("токен HuggingFace на месте, длина", len(token))
except Exception as e:
    print("ТОКЕНА НЕТ:", e)
    print("Add-ons -> Secrets -> HF_TOKEN. Без него скачивание весов "
          "режется по скорости.")

## 2. Библиотеки

vLLM вместо `model.generate`: на пятнадцати тысячах вопросов разница
в скорости — порядок. `sentence-transformers` нужен третьему фильтру
(кросс-энкодер-учитель).

Установка vLLM тянет свою сборку torch с CUDA — это два-четыре гигабайта
и **пять-пятнадцать минут**. Дольше двадцати — это обычно не скачивание,
а перебор версий зависимостей; он крутится часами, а ускоритель всё это
время включён и квота идёт. Поэтому вывод не глушится, и ячейка печатает
потраченное время числом.

In [ ]:
# Без --progress-bar off Kaggle заливает вывод тысячами строк, а с -q
# не видно ничего вовсе — и это хуже. Отличить «качает три гигабайта»
# от «pip перебирает версии зависимостей» можно только по выводу,
# а перебор крутится часами, и всё это время ускоритель включён
# и квота идёт. Поэтому вывод оставлен, а время печатается числом:
# пять-пятнадцать минут — норма, больше двадцати — Run -> Interrupt.
import subprocess, sys, time

t = time.time()
for пакет in ('vllm', 'sentence-transformers'):
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-U', пакет,
                    '--progress-bar', 'off'], check=True)
print(f'установка заняла {(time.time() - t) / 60:.1f} мин')

import vllm
print('vllm', vllm.__version__)


## 3. Код и данные на место

In [ ]:
import glob, os, shutil, zipfile

# Kaggle монтирует датасеты по-разному: то прямо в /kaggle/input/<имя>,
# то глубже, внутри /kaggle/input/datasets/<владелец>/<имя>. Поэтому путь
# не задаётся, а источник ищется по содержимому: нужен каталог, где лежит
# пакет rufin или корпус. Глубина ограничена, чтобы не обходить всё подряд.
def find_source(root='/kaggle/input', max_depth=4):
    found, archives = [], []
    for cur, dirs, files in os.walk(root):
        if cur.count('/') - root.count('/') >= max_depth:
            dirs.clear()
        if 'rufin' in dirs:
            stale = ('__notebook__.ipynb' in files or 'embeddings' in dirs
                     or '__output__.json' in files)
            found.append((1 if stale else 0, cur))
        archives += [os.path.join(cur, f) for f in files if f.endswith('.zip')]
    if found:
        found.sort()
        return found[0][1], None
    return (os.path.dirname(archives[0]), archives[0]) if archives else (None, None)


def show_tree(root='/kaggle/input', max_depth=3):
    for cur, dirs, files in os.walk(root):
        depth = cur.count('/') - root.count('/')
        if depth >= max_depth:
            dirs.clear()
        print('   ' * depth + os.path.basename(cur) + '/')
        for f in sorted(files)[:6]:
            print('   ' * (depth + 1) + f)


src, archive = find_source()
if src is None:
    print('что ноутбук видит на входе:')
    show_tree()
    raise SystemExit('датасет не подключён: панель справа -> + Add Input -> ru-fin-retrieval -> +')

print('источник:', src)
if archive:
    print('лежит архивом, распаковываю:', archive)
    with zipfile.ZipFile(archive) as z:
        z.extractall('/kaggle/working')
else:
    for item in glob.glob(os.path.join(src, '*')):
        dst = os.path.join('/kaggle/working', os.path.basename(item))
        if os.path.isdir(item):
            shutil.copytree(item, dst, dirs_exist_ok=True)
        else:
            shutil.copy2(item, dst)

%cd /kaggle/working
print('\nв рабочей папке:', sorted(os.listdir('.')))

assert os.path.isdir('rufin'), f'пакета rufin нет; в источнике было: {sorted(os.listdir(src))}'
# Kaggle иногда разжимает архив при создании датасета, имя корпуса плавает
assert glob.glob('acts.jsonl*'), 'корпуса нет'
print('всё на месте:', sorted(glob.glob('acts.jsonl*')))


In [ ]:
import glob
# сплит обязателен: без него генерация уйдёт на акты теста
print('сплит:', glob.glob('/kaggle/input/**/split.json', recursive=True)
      or glob.glob('split.json'))
print('запросы:', glob.glob('/kaggle/input/**/queries.jsonl', recursive=True)
      or glob.glob('queries.jsonl'))
print('пул:', glob.glob('/kaggle/input/**/pool_candidates.tsv', recursive=True)
      or glob.glob('pool_candidates.tsv'))

### 3б. Каноническая нарезка на место. Обязательно

Нарезка детерминирована, но зависит от версии токенизатора — а установка
vLLM тянет свой `transformers` и границы сдвигает: **61 922 фрагмента
вместо 62 594** на том же корпусе. Идентификатор фрагмента позиционный
(`акт#номер`), поэтому при сдвиге он не исчезает, а начинает указывать
на другой текст. Эталон сгенерированных вопросов съехал бы молча,
и обучающая выборка учила бы модель находить не то.

Поэтому нарезка не пересчитывается, а берётся готовая — та же, по которой
считались выдачи и разметка. Ячейка должна напечатать **62594**.

Кэш сносится первой строкой: упавший прогон успевает записать туда свою
нарезку, а готовая берётся раньше, чем считается новая. Без сноса
неканоническая нарезка переживает любое число перезапусков и печатает
бодрое «взята готовая».


In [ ]:
import glob, gzip, os, shutil

# Кэш сносится первым, и это не перестраховка. Упавший прогон успевает
# посчитать нарезку и записать её сюда, а `build_chunks_cached` берёт
# готовую раньше, чем считает новую, — и неканоническая нарезка
# переживает любое число перезапусков, печатая «взята готовая».
dst = '/kaggle/working/chunks/base.jsonl'
if os.path.exists(dst):
    было = sum(1 for _ in open(dst, encoding='utf-8'))
    print(f'сношу кэш: {было} фрагментов')
    os.remove(dst)

найдено = (glob.glob('/kaggle/input/**/chunks_base.jsonl', recursive=True)
           + glob.glob('/kaggle/input/**/chunks_base.jsonl.gz', recursive=True))
assert найдено, 'нет chunks_base.jsonl: подключите датасет ru-fin-queries'
src = найдено[0]

os.makedirs('/kaggle/working/chunks', exist_ok=True)
opener = gzip.open if src.endswith('.gz') else open
with opener(src, 'rb') as fi, open(dst, 'wb') as fo:
    shutil.copyfileobj(fi, fo, length=1 << 20)

n = sum(1 for _ in open(dst, encoding='utf-8'))
print('источник:', src)
print('фрагментов:', n)
assert n == 62594, f'ожидалось 62594, а не {n}: нарезка не каноническая'


## 4. Замер скорости

Сотня фрагментов, три стиля. Считает, сколько вопросов в секунду даёт
модель и во что это выльется на десяти и пятнадцати тысячах. **Объём
обучающей выборки назначается по этому числу, а не по плану:** десять
тысяч за три часа лучше пятнадцати, которые не досчитаются к концу
сессии.

Двадцать минут, включая скачивание весов.

In [ ]:
!cd /kaggle/working && python run_phase_c.py --probe 100

## 5. Пробный прогон

Сто актов, по двадцать вопросов на выборку. Проверяем, что генерация,
фильтры и запись работают, и смотрим на сами вопросы глазами. Десять
минут — дешевле, чем узнать о браке через три часа.

In [ ]:
!cd /kaggle/working && python run_phase_c.py \
    --limit-acts 100 --probe 0 --train 20 --dev 20 --test 20 \
    --out /kaggle/working/proba

import json
for имя in ('synthetic_train', 'synthetic_dev', 'synthetic_test'):
    path = f'/kaggle/working/proba/queries/{имя}.jsonl'
    try:
        rows = [json.loads(l) for l in open(path, encoding='utf-8')]
    except FileNotFoundError:
        print(имя, '— файла нет'); continue
    print(f'\n=== {имя}: {len(rows)} ===')
    for r in rows[:6]:
        print(f"  [{r['style']}] {r['text'][:110]}")

## 6. Полный прогон

Объём в `--train` подставляется **по замеру из шага 4**. Третий фильтр
(`--teacher-filter`) считается отдельно и стоит дорого: `вопросов ×
глубина` проходов кросс-энкодера. Если по замеру видно, что в сессию
не укладывается, сначала прогоняются генерации, а фильтр — вторым
запуском: шаги независимы и готовое не пересчитывают.

**Про кнопку Save Version — до того, как она понадобится.**
**Save & Run All (Commit)** не сохраняет текущую сессию, а запускает
тетрадь заново с нуля: рабочий каталог чистится, все ячейки идут
сверху донизу. Нажать её после долгого ручного прогона — значит
выбросить его и переплатить квотой. Она годится только как способ
запустить счёт с самого начала, не держа вкладку открытой,
и решать это надо **до** запуска, а не после.

**Quick Save** ничего не перезапускает, но по умолчанию **не сохраняет
файлы**: нужно раскрыть **Advanced settings** и включить сохранение
вывода.

Поэтому порядок такой: **сначала скачать файлы (шаг 9), потом
сохранять версию.** Всё нужное маленькое.

In [ ]:
# Обучающая выборка и dev: модель-генератор, глубина фильтра 30.
# Глубина 30 вместо 100 — решение по остатку квоты (14 ч 24 м на 29.09):
# фильтр дешевеет с 4-5 ч до 1,3 ч, и это оставляет больше восьми часов
# запаса, то есть право переделать любой прогон дважды.
!cd /kaggle/working && python run_phase_c.py --probe 0 \
    --train 12000 --dev 300 \
    --teacher-filter --filter-depth 30 \
    --out /kaggle/working/gen


### 6б. Тест — моделью другого семейства

Отдельным запуском, и модель обязательно из другого семейства: иначе
тест померит не качество поиска, а то, насколько ученик выучил стиль
своего же генератора. Держать две модели в памяти незачем.


In [ ]:
!cd /kaggle/working && python run_phase_c.py --probe 0 \
    --test 400 --test-model TheBloke/Mistral-7B-Instruct-v0.2-AWQ \
    --out /kaggle/working/test


## 7. Разметка пула судьёй. Последним прогоном

Судья идёт **после** пересчёта выдач, а не до него, хотя в задании
он стоит раньше. Причина видна на `make pool`: пул собирается по
выдачам всех конфигураций, а выдачи посчитаны по прежним 211 запросам.
У 180 живых вопросов из 228, которым нужен эталон, выдачи нет вовсе,
и `make pool` отказывается собирать лист — пул по неполным выдачам
занижает те конфигурации, которых в нём нет, а по метрикам этого
не видно.

Поэтому порядок такой: этап B по нынешнему набору запросов (плотный
поиск и BM25 — без ускорителя, бесплатно; реранкер — на карте), затем
локально `make pool`, затем датасет запросов перезаливается вместе
с `pool_candidates.tsv`, и только тогда — эта ячейка.


In [ ]:
!cd /kaggle/working && python run_phase_c.py --probe 0 --judge

## 8. Что получилось

In [ ]:
import json
r = json.load(open('/kaggle/working/report_phase_c.json', encoding='utf-8'))
for шаг, v in r.items():
    print(f'--- {шаг}')
    print('   ', v if not isinstance(v, dict) else
          {k: x for k, x in v.items() if not isinstance(x, dict)})

## 9. Скачать файлы. **До** всякого Save Version

В интерактивной сессии вывод не публикуется сам: `/kaggle/working`
живёт, пока жива сессия. **Save & Run All (Commit)** его не спасает,
а уничтожает — он чистит каталог и считает всё заново. **Quick Save**
без **Advanced settings** файлы тоже не сохраняет.

Поэтому сначала ячейка ниже и ссылка под ней, и только потом кнопки.

Что куда класть на маке:

* `queries/synthetic_train.jsonl`, `synthetic_dev.jsonl`,
  `synthetic_test.jsonl`, `judge.tsv` → `data/queries/`;
* `report_phase_c.json` → `docs/raw/`.

Дальше: `make queries`, `make judge`, `make sample-for-human`,
`make kappa`, `make bench`.


In [ ]:
import os, shutil
from IPython.display import FileLink

# Складываем всё мелкое в один архив: вывозить по файлу из панели Output
# дольше и легко забыть половину. Корпус и веса не трогаем — они большие
# и на маке не нужны.
os.makedirs('/kaggle/working/vyvoz', exist_ok=True)
взято = []
for корень in ('/kaggle/working/gen', '/kaggle/working/test',
               '/kaggle/working/judge', '/kaggle/working'):
    for база, _, файлы in os.walk(корень):
        if 'chunks' in база or 'vyvoz' in база:
            continue
        for имя in файлы:
            if имя.endswith(('.jsonl', '.tsv')) or имя == 'report_phase_c.json':
                путь = os.path.join(база, имя)
                if os.path.getsize(путь) > 200 * 1024 * 1024:
                    continue
                shutil.copy2(путь, os.path.join('/kaggle/working/vyvoz', имя))
                взято.append((имя, os.path.getsize(путь) // 1024))

for имя, кб in sorted(set(взято)):
    print(f'   {имя:<28} {кб:>8} КБ')
assert взято, 'нечего вывозить: генерация ещё не давала файлов'

shutil.make_archive('/kaggle/working/vyvoz_etap_c', 'zip', '/kaggle/working/vyvoz')
print('\nархив:', os.path.getsize('/kaggle/working/vyvoz_etap_c.zip') // 1024, 'КБ')
FileLink('vyvoz_etap_c.zip')
